In [3]:
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix,balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import mlflow
import lightgbm as lgb

In [2]:
mlflow.set_experiment("binary_classification_experiment_class_mapping_changed_v2")

<Experiment: artifact_location='file:d:/fourth_year/graduation_project/JGuard/defenders/mult-iturn/NBF/notebooks/mlruns/2', creation_time=1780748780088, experiment_id='2', last_update_time=1780748780088, lifecycle_stage='active', name='binary_classification_experiment_class_mapping_changed_v2', tags={}, trace_location=None, workspace='default'>

In [3]:
def log_trained_model(model,y_pred,y_test,features,params: dict,run_name: str = "run",\
        model_name: str = "model",plot_name: str = "confusion_matrix.png",\
            dim_reduction_model=None):
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("features", str(features))

        # Metrics
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        f1_score_val = f1_score(y_test, y_pred, average='macro')
        classification_report_val = classification_report(y_test, y_pred, output_dict=True)
        mlflow.log_metric("balanced_accuracy", bal_acc)
        mlflow.log_metric("f1_score", f1_score_val)
        mlflow.log_dict(classification_report_val, "classification_report.json")
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.savefig(plot_name)
        plt.close()
        mlflow.log_artifact(plot_name)

        mlflow.sklearn.log_model(model, model_name)
        if dim_reduction_model is not None:
            mlflow.sklearn.log_model(dim_reduction_model, f"{model_name}_dim_reduction")

        print(f"Run complete. Balanced Accuracy: {bal_acc:.4f}")

        return {
            "balanced_accuracy": bal_acc,
            "f1_score": f1_score_val,
            "confusion_matrix": cm.tolist()  
        }

In [4]:
data=torch.load("./../train_data_with_context/all_conversations.pt")
data2=torch.load("./../train_data_with_context/all_conversations2.pt")

In [5]:
data[0][0].keys()

dict_keys(['x_t', 'zt', 'score', 'y'])

In [6]:
labels_map={
    1:0,
    2:0,
    3:0,
    4:1,
    5:1
}

In [7]:
X = []
y=[]
for convo in data:
    for turn in convo:
        X.append(torch.concat([turn["x_t"], turn["zt"]], dim=0))
        y.append(labels_map[turn["score"]])

In [8]:
X2 = []
y2=[]
for convo in data2:
    for turn in convo:
        X2.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))
        y2.append(labels_map[turn["score"]])

In [9]:
from collections import Counter
c=Counter(y)
c

Counter({0: 23007, 1: 5972})

In [10]:
# high imbalance ratio
ratio=c[0]/c[1]
ratio

3.852478231748158

In [11]:
X = np.array(X)
y = np.array(y)
X2 = np.array(X2)
y2 = np.array(y2)

In [12]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

In [13]:
# dimensionality reduction using PCA on the first dataset xt and zt features
pca1 = PCA(n_components=0.95, random_state=42)
X_train1_pca = pca1.fit_transform(X_train1)
X_test1_pca = pca1.transform(X_test1)
print("Number of components:", pca1.n_components_)
print("Total explained variance:", pca1.explained_variance_ratio_.sum())


# dimensionality reduction using PCA on the second dataset xt and ut features
pca2 = PCA(n_components=0.95, random_state=42)
X_train2_pca = pca2.fit_transform(X_train2)
X_test2_pca = pca2.transform(X_test2)
print("Number of components:", pca2.n_components_)
print("Total explained variance:", pca2.explained_variance_ratio_.sum())

Number of components: 213
Total explained variance: 0.95042664
Number of components: 352
Total explained variance: 0.9501169


In [15]:
svm_balanced = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")

y_train_pred = svm_balanced.fit(X_train1, y_train1).predict(X_train1)
y_test_pred = svm_balanced.predict(X_test1)

In [16]:
print("train_f1_score:", f1_score(y_train1, y_train_pred, average='macro'))
print("Balanced f1_score:", f1_score(y_test1, y_test_pred, average='macro'))
print(classification_report(y_test1, y_test_pred))

train_f1_score: 0.842369945302033
Balanced f1_score: 0.7775008769094206
              precision    recall  f1-score   support

           0       0.95      0.83      0.88      4602
           1       0.56      0.84      0.67      1194

    accuracy                           0.83      5796
   macro avg       0.75      0.83      0.78      5796
weighted avg       0.87      0.83      0.84      5796



In [17]:
log_trained_model(svm_balanced,y_test_pred,y_test1,features="Xt_Zt",\
                  params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},\
                    run_name="SVM_rbf_balanced",model_name="svm_rbf_different",plot_name="svm_rbf_balanced_cm.png")

2026/06/06 15:56:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:56:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:56:48 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmp8_06a19j\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.8346


{'balanced_accuracy': 0.8346485797086257,
 'f1_score': 0.7775008769094206,
 'confusion_matrix': [[3797, 805], [186, 1008]]}

# with dimensionality reduction

In [18]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")  

y_train_pred=model.fit(X_train1_pca, y_train1).predict(X_train1_pca)
y_test_pred = model.predict(X_test1_pca)

In [19]:
print(classification_report(y_test1, y_test_pred))

              precision    recall  f1-score   support

           0       0.95      0.82      0.88      4602
           1       0.55      0.84      0.66      1194

    accuracy                           0.83      5796
   macro avg       0.75      0.83      0.77      5796
weighted avg       0.87      0.83      0.84      5796



In [20]:
log_trained_model(run_name="SVM_pca_different_weights", model=model, y_pred=y_test_pred, y_test=y_test1, features="PCAXt_Zt", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_different", plot_name="svm_rbf_different_cm.png",
                dim_reduction_model=pca1)

2026/06/06 15:58:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:58:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:58:41 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpnl3y39dz\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/06/06 15:58:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:58:41 WARNING mlflow.sklearn: Savi

Run complete. Balanced Accuracy: 0.8301


{'balanced_accuracy': 0.8300875302195463,
 'f1_score': 0.7729533514658258,
 'confusion_matrix': [[3782, 820], [193, 1001]]}

# trial 2

In [21]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced") 

# Train
y_train_pred =model.fit(X_train2, y_train2).predict(X_train2)

# Predict
y_test_pred = model.predict(X_test2)

# Evaluate
print("Accuracy:", accuracy_score(y_test2, y_test_pred))
print(classification_report(y_test2, y_test_pred))

Accuracy: 0.839544513457557
              precision    recall  f1-score   support

           0       0.95      0.84      0.89      4602
           1       0.58      0.82      0.68      1194

    accuracy                           0.84      5796
   macro avg       0.76      0.83      0.79      5796
weighted avg       0.87      0.84      0.85      5796



In [22]:
log_trained_model(run_name="SVM_balanced_weights", model=model, y_pred=y_test_pred, y_test=y_test2, features="Xt_ut", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_balanced", plot_name="svm_rbf_balanced_cm.png")

2026/06/06 16:14:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 16:14:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 16:15:01 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmp06qhau5d\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.8335


{'balanced_accuracy': 0.8335233315643843,
 'f1_score': 0.785960840571902,
 'confusion_matrix': [[3883, 719], [211, 983]]}

In [23]:
# Create SVM model
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")  

# Train
model.fit(X_train2_pca, y_train2)

# Predict
y_test_pred = model.predict(X_test2_pca)

# Evaluate
print("Accuracy:", accuracy_score(y_test2, y_test_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test2, y_test_pred))
print("F1 Score:", f1_score(y_test2, y_test_pred, average='macro'))
print(classification_report(y_test2, y_test_pred))

Accuracy: 0.8378191856452726
Balanced Accuracy: 0.8305761750953813
F1 Score: 0.7835596137487937
              precision    recall  f1-score   support

           0       0.95      0.84      0.89      4602
           1       0.57      0.82      0.68      1194

    accuracy                           0.84      5796
   macro avg       0.76      0.83      0.78      5796
weighted avg       0.87      0.84      0.85      5796



In [27]:
import joblib
joblib.dump(model, "svm_model_v2_45mapping.pkl")

['svm_model_v2_45mapping.pkl']

In [24]:
log_trained_model(run_name="SVM_balanced_weights_pca", model=model, y_pred=y_test_pred, y_test=y_test2, features="pca(Xt_ut)", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_balanced", plot_name="cm.png", dim_reduction_model=pca2)

2026/06/06 16:18:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 16:18:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 16:18:31 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpyj3d5zmx\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/06/06 16:18:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 16:18:31 WARNING mlflow.sklearn: Savi

Run complete. Balanced Accuracy: 0.8306


{'balanced_accuracy': 0.8305761750953813,
 'f1_score': 0.7835596137487937,
 'confusion_matrix': [[3879, 723], [217, 977]]}

# random forest

In [14]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Parameter distributions
param_dist = {
    "n_estimators": randint(200, 800),
    "max_depth": [None, 10, 20, 30, 40, 50],
    "min_samples_split": randint(2, 15),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy"]
}

# Random search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,           
    scoring="f1",         
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Train
random_search.fit(X_train1_pca, y_train1)

# Best model
best_rf = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

# Predict
y_test_pred = best_rf.predict(X_test1_pca)

print("\nF1 Score:", f1_score(y_test1, y_test_pred))
print(classification_report(y_test1, y_test_pred))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters:
{'bootstrap': False, 'criterion': 'entropy', 'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 6, 'min_samples_split': 7, 'n_estimators': 305}

F1 Score: 0.6119631901840491
              precision    recall  f1-score   support

           0       0.91      0.87      0.89      4602
           1       0.56      0.67      0.61      1194

    accuracy                           0.83      5796
   macro avg       0.74      0.77      0.75      5796
weighted avg       0.84      0.83      0.83      5796



In [15]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Parameter distributions
param_dist = {
    "n_estimators": randint(200, 800),
    "max_depth": [None, 10, 20, 30, 40, 50],
    "min_samples_split": randint(2, 15),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy"]
}

# Random search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,           
    scoring="f1",         
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Train
random_search.fit(X_train2_pca, y_train1)

# Best model
best_rf = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

y_test_pred = best_rf.predict(X_test2_pca)

print("\nF1 Score:", f1_score(y_test1, y_test_pred))
print(classification_report(y_test1, y_test_pred))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters:
{'bootstrap': True, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 9, 'min_samples_split': 9, 'n_estimators': 328}

F1 Score: 0.5689497716894977
              precision    recall  f1-score   support

           0       0.88      0.92      0.90      4602
           1       0.63      0.52      0.57      1194

    accuracy                           0.84      5796
   macro avg       0.75      0.72      0.73      5796
weighted avg       0.83      0.84      0.83      5796

